# Task 100: sme_stellar — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Stellar spectroscopy parameter fitting using SME

**Paper**: Spectroscopy Made Easy: A New Tool for Fitting Observations with Synthetic Spectra
**Repository**: https://github.com/AWehrhahn/SME

### Physical Background

Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.

### Forward Model

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

### Inverse Problem

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

### Performance Metrics
- **PSNR**: 63.25 dB
- **SSIM**: N/A (1D spectrum)
- **Evaluation**: Stellar spectrum fit + parameter recovery (PSNR + CC + RE)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
sme_stellar - Stellar Spectral Synthesis & Parameter Fitting
=============================================================
From a high-resolution stellar spectrum, fit stellar parameters
(T_eff, log g, [Fe/H]) and element abundances.

Physics:
  - Forward model: Planck continuum × Voigt absorption lines
  - 5 elements (Fe, Ca, Mg, Na, Ti), each with 2–3 characteristic lines
  - Line depth depends on abundance, width on T_eff / log_g
  - Inverse: scipy.optimize.differential_evolution
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`planck_continuum`, `voigt_line`, `pack_params`, `unpack_params`, `objective`, `main`

In [ ]:
# ====================================================================
# 1. Forward model
# ====================================================================
def planck_continuum(wavelength_AA, T_eff):
    """Planck function as continuum (normalised, in wavelength units)."""
    h = 6.626e-34
    c = 3.0e8
    k = 1.381e-23
    lam_m = wavelength_AA * 1e-10
    B = (2.0 * h * c**2 / lam_m**5) / (np.exp(h * c / (lam_m * k * T_eff)) - 1.0)
    return B / B.max()

def voigt_line(wavelength, lam0, sigma_G, gamma_L, depth):
    """Single Voigt absorption line."""
    delta = wavelength - lam0
    profile = voigt_profile(delta, sigma_G, gamma_L)
    profile = profile / profile.max() if profile.max() > 0 else profile
    return depth * profile

# ====================================================================
# 3. Inverse: differential evolution
# ====================================================================
def pack_params(T_eff, log_g, feh, abundances):
    """Pack parameters into a 1D vector."""
    vec = [T_eff, log_g, feh]
    for elem in sorted(abundances.keys()):
        vec.append(abundances[elem])
    return np.array(vec)

def unpack_params(vec):
    """Unpack 1D vector into parameters."""
    T_eff = vec[0]
    log_g = vec[1]
    feh   = vec[2]
    elements = sorted(GT_ABUNDANCES.keys())
    abundances = {elem: vec[3 + i] for i, elem in enumerate(elements)}
    return T_eff, log_g, feh, abundances

def objective(vec, wavelength, flux_obs):
    """Chi-squared objective."""
    T_eff, log_g, feh, abundances = unpack_params(vec)
    flux_model = synthesize_spectrum(wavelength, T_eff, log_g, feh, abundances)
    residual = flux_obs - flux_model
    return np.sum(residual**2)

# ====================================================================
# 6. Main
# ====================================================================
def main():
    print("=" * 60)
    print("Task 100: Stellar Spectral Synthesis (sme_stellar)")
    print("=" * 60)

    t0 = time.time()

    # Generate data
    print("\n[1] Generating ground-truth spectrum ...")
    flux_gt, flux_obs = generate_gt_data()

    # Inverse solve
    print("[2] Fitting stellar parameters (differential evolution) ...")
    T_fit, logg_fit, feh_fit, ab_fit, flux_fit = solve_inverse(WAVELENGTH, flux_obs)

    elapsed = time.time() - t0
    print(f"    Elapsed: {elapsed:.1f} s")

    # Parameters
    gt_params = (GT_TEFF, GT_LOGG, GT_FEH, GT_ABUNDANCES)
    fit_params = (T_fit, logg_fit, feh_fit, ab_fit)

    # Metrics
    print("[3] Computing metrics ...")
    psnr, cc, p_errors = compute_metrics(flux_gt, flux_fit, gt_params, fit_params)

    print(f"    Spectrum PSNR = {psnr:.2f} dB")
    print(f"    Spectrum CC   = {cc:.6f}")
    print(f"    T_eff: GT={GT_TEFF:.0f} K, Fit={T_fit:.0f} K, RE={p_errors['RE_T_eff']:.4f}")
    print(f"    log_g: GT={GT_LOGG:.2f}, Fit={logg_fit:.2f}, RE={p_errors['RE_log_g']:.4f}")
    print(f"    [Fe/H]: GT={GT_FEH:.2f}, Fit={feh_fit:.2f}")
    for elem in sorted(GT_ABUNDANCES.keys()):
        print(f"    [{elem}/H]: GT={GT_ABUNDANCES[elem]:.2f}, Fit={ab_fit[elem]:.2f}, AE={p_errors[f'AE_{elem}']:.4f}")

    # Build metrics dict
    metrics = {
        "PSNR": float(psnr),
        "CC": float(cc),
        "RE_Teff": float(p_errors["RE_T_eff"]),
        "RE_logg": float(p_errors["RE_log_g"]),
    }
    for elem in sorted(GT_ABUNDANCES.keys()):
        metrics[f"AE_{elem}"] = float(p_errors[f"AE_{elem}"])

    # Save
    print("[4] Saving outputs ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), flux_gt)
        np.save(os.path.join(d, "recon_output.npy"), flux_fit)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    # Visualization
    print("[5] Plotting ...")
    extra_info = dict(p_errors)
    extra_info["psnr"] = psnr
    extra_info["cc"] = cc
    plot_results(WAVELENGTH, flux_gt, flux_obs, flux_fit,
                 gt_params, fit_params, extra_info)

    print(f"\n{'='*60}")
    print("Task 100 COMPLETE")
    print(f"{'='*60}")
    return metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

Functions: `synthesize_spectrum`, `generate_gt_data`

In [ ]:
def synthesize_spectrum(wavelength, T_eff, log_g, feh, abundances):
    """
    Synthesize a stellar spectrum.
    
    Parameters
    ----------
    wavelength : array, Angstroms
    T_eff : float, K
    log_g : float
    feh  : float, [Fe/H]
    abundances : dict  {element: [X/H]}
    
    Returns
    -------
    flux : array, normalised flux
    """
    continuum = planck_continuum(wavelength, T_eff)
    absorption = np.zeros_like(wavelength)

    for elem, lam0, gf in LINE_LIST:
        ab = abundances.get(elem, 0.0) + feh
        # Gaussian width depends on T_eff (thermal broadening)
        sigma_thermal = lam0 / 3e5 * np.sqrt(2.0 * 1.381e-23 * T_eff / (56.0 * 1.66e-27))
        sigma_G = max(sigma_thermal, 0.02)
        # Lorentzian width depends on log_g (pressure broadening)
        gamma_L = 0.05 * 10.0**(0.3 * (4.5 - log_g))
        # Line depth depends on abundance and oscillator strength
        depth = 0.3 * 10.0**(gf + ab) / (1.0 + 10.0**(gf + ab))
        depth = np.clip(depth, 0.0, 0.95)
        absorption += voigt_line(wavelength, lam0, sigma_G, gamma_L, depth)

    flux = continuum * (1.0 - np.clip(absorption, 0, 0.99))
    return flux

# ====================================================================
# 2. Generate ground-truth data
# ====================================================================
def generate_gt_data():
    """Generate GT spectrum + noisy observation."""
    flux_gt = synthesize_spectrum(WAVELENGTH, GT_TEFF, GT_LOGG, GT_FEH, GT_ABUNDANCES)
    noise_level = flux_gt.mean() / SNR
    noise = np.random.normal(0, noise_level, N_WAVE)
    flux_obs = flux_gt + noise
    return flux_gt, flux_obs

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

Functions: `solve_inverse`

In [ ]:
def solve_inverse(wavelength, flux_obs):
    """Fit stellar parameters using differential evolution."""
    bounds = [
        (4500, 7000),   # T_eff
        (3.0, 5.5),     # log_g
        (-1.0, 0.5),    # [Fe/H]
    ]
    for _ in sorted(GT_ABUNDANCES.keys()):
        bounds.append((-0.5, 0.5))  # [X/H]

    result = differential_evolution(
        objective, bounds, args=(wavelength, flux_obs),
        seed=123, maxiter=300, tol=1e-8, popsize=20,
        mutation=(0.5, 1.5), recombination=0.9,
        polish=True
    )
    T_eff, log_g, feh, abundances = unpack_params(result.x)
    flux_fit = synthesize_spectrum(wavelength, T_eff, log_g, feh, abundances)
    return T_eff, log_g, feh, abundances, flux_fit

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `plot_results`

In [ ]:
# ====================================================================
# 4. Metrics
# ====================================================================
def compute_metrics(flux_gt, flux_fit, gt_params, fit_params):
    """PSNR, CC, and parameter relative errors."""
    # Spectrum metrics
    mse = np.mean((flux_gt - flux_fit)**2)
    psnr = 10.0 * np.log10(flux_gt.max()**2 / mse) if mse > 0 else 100.0
    cc = float(np.corrcoef(flux_gt, flux_fit)[0, 1])

    # Parameter relative errors
    param_names = ["T_eff", "log_g", "[Fe/H]"]
    gt_vals = [gt_params[0], gt_params[1], gt_params[2]]
    fit_vals = [fit_params[0], fit_params[1], fit_params[2]]
    
    param_errors = {}
    for name, gv, fv in zip(param_names, gt_vals, fit_vals):
        if abs(gv) > 1e-6:
            param_errors[f"RE_{name}"] = abs(fv - gv) / abs(gv)
        else:
            param_errors[f"RE_{name}"] = abs(fv - gv)

    # Abundance errors
    gt_ab = gt_params[3]
    fit_ab = fit_params[3]
    for elem in sorted(gt_ab.keys()):
        param_errors[f"AE_{elem}"] = abs(fit_ab[elem] - gt_ab[elem])

    return psnr, cc, param_errors

# ====================================================================
# 5. Visualization
# ====================================================================
def plot_results(wavelength, flux_gt, flux_obs, flux_fit,
                 gt_params, fit_params, param_errors):
    """4-panel figure."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Panel 1: full spectrum overlay
    ax = axes[0, 0]
    ax.plot(wavelength, flux_obs, 'gray', alpha=0.4, lw=0.5, label='Observed')
    ax.plot(wavelength, flux_gt, 'b-', lw=1.0, label='GT spectrum')
    ax.plot(wavelength, flux_fit, 'r--', lw=1.0, label='Fitted spectrum')
    ax.set_xlabel("Wavelength (Å)")
    ax.set_ylabel("Normalised Flux")
    ax.set_title("Stellar Spectrum: GT vs Fitted")
    ax.legend(fontsize=8)

    # Panel 2: zoom on Na D lines
    ax = axes[0, 1]
    mask = (wavelength > 5870) & (wavelength < 5920)
    ax.plot(wavelength[mask], flux_obs[mask], 'gray', alpha=0.5, lw=0.8, label='Observed')
    ax.plot(wavelength[mask], flux_gt[mask], 'b-', lw=1.2, label='GT')
    ax.plot(wavelength[mask], flux_fit[mask], 'r--', lw=1.2, label='Fitted')
    ax.set_xlabel("Wavelength (Å)")
    ax.set_ylabel("Normalised Flux")
    ax.set_title("Zoom: Na D doublet (5890/5896 Å)")
    ax.legend(fontsize=8)

    # Panel 3: residuals
    ax = axes[1, 0]
    residual = flux_gt - flux_fit
    ax.plot(wavelength, residual, 'k-', lw=0.5)
    ax.axhline(0, color='r', ls='--', lw=0.5)
    ax.set_xlabel("Wavelength (Å)")
    ax.set_ylabel("Residual (GT - Fit)")
    ax.set_title(f"Residuals | PSNR={param_errors['psnr']:.1f} dB, CC={param_errors['cc']:.4f}")

    # Panel 4: parameter comparison bar chart
    ax = axes[1, 1]
    labels = ["T_eff/1000", "log_g", "[Fe/H]+1"]
    gt_v = [gt_params[0]/1000, gt_params[1], gt_params[2]+1]
    fit_v = [fit_params[0]/1000, fit_params[1], fit_params[2]+1]
    x = np.arange(len(labels))
    ax.bar(x - 0.15, gt_v, 0.3, label='GT', color='steelblue')
    ax.bar(x + 0.15, fit_v, 0.3, label='Fitted', color='salmon')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title("Stellar Parameters")
    ax.legend()

    plt.tight_layout()
    for path in [os.path.join(RESULTS_DIR, "vis_result.png"),
                 os.path.join(ASSETS_DIR, "vis_result.png")]:
        fig.savefig(path, dpi=150)
    plt.close(fig)

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = main()

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **sme_stellar**:

1. **Problem Setup**: Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=63.25 dB, SSIM=N/A (1D spectrum))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Spectroscopy Made Easy: A New Tool for Fitting Observations with Synthetic Spectra
- Repository: https://github.com/AWehrhahn/SME
